This experiment runs Microgrid consisting of PV source, Battery system and Load consumption.

The system is controlled reactively.

Operation Metrics:
...


In [1]:
import pandas as pd

In [2]:
from smart_grid_lab.core import (
    Time,
    PV,
    Load,
    Battery,
    SetUp,
    Simulation,
    Results,
)

from smart_grid_lab.core.controllers import Reactive

from smart_grid_lab.domain.microgrid_configuration import (
    MICROGRID_LOCATION,
    PV_AREA_M2,
    BatteryConfig,
)

from smart_grid_lab.domain.builders import (
    build_battery,
)

from smart_grid_lab.application.load_template import (
    sample_pv_kW,
    sample_load_kW,
)

In [3]:
from smart_grid_lab.application.use_cases.pv_load_battery import run_simulation

In [4]:
from smart_grid_lab.ui_dash.plot_utils import build_figure

# Configure Experiment Location and Time 

In [5]:
time = Time(
    start=pd.Timestamp(year=2016, month=3, day=15, hour=12, tz="utc"),
    end=pd.Timestamp(year=2016, month=3, day=30, hour=12, tz="utc"),
    step=pd.Timedelta(15, "min"),
)

# Load Dataset

In [6]:
pv_sample = sample_pv_kW(time)

In [7]:
load_sample = sample_load_kW(time)

# Configure Components

In [8]:
battery_config = BatteryConfig(
    capacity_kwh = 100.0,
    max_charge_kw = 30.0,
    max_discharge_kw = 30.0,
    max_charge_efficiency = 0.95,
    max_discharge_efficiency = 0.95,
    initial_soc = 0.5)
battery = build_battery(battery_config)

# Build Controller

In [9]:
controller = Reactive()

# Run Simulation

In [10]:
results = run_simulation(
    time, pv_sample, load_sample, battery, controller, use_bar=True
)

100%|██████████| 1440/1440 [00:00<00:00, 18653.96it/s]


# Examine Results

In [11]:
results.trajectory.head()

,pv_power,load_power,battery_available_power,battery_charge_power,battery_discharge_power,battery_soc
timestamp,,,,,,
2016-03-15 12:00:00+00:00,2.314,24.014,0.000,0.0,0.000,0.500000
2016-03-15 12:15:00+00:00,1.727,24.703,-21.700,0.0,21.700,0.448497
2016-03-15 12:30:00+00:00,1.366,23.543,-22.976,0.0,22.976,0.393951
2016-03-15 12:45:00+00:00,1.359,23.896,-22.177,0.0,22.177,0.341293
2016-03-15 13:00:00+00:00,1.087,23.267,-22.537,0.0,22.537,0.287775


## Plot Microgrid Telemetry

In [14]:
build_figure(results.trajectory, 
                       ["pv_power", 
                        "load_power", 
                        "battery_available_power",
                        "battery_charge_power",
                        "battery_discharge_power"],
                       ["battery_soc"])

## Display Metrics Table

In [13]:
results.metrics

,pv_energy_kwh,load_energy_kwh,electrolyser_energy_kwh,hydrogen_produced_kg,final_tank_level,final_battery_soc,electrolyser_degradation
0,938.857,6088.107,None,None,None,0.05,None
